# The `object` Class — Advanced Problems with Fully Worked Solutions
## A slow, experiment-driven tutorial

This notebook is a **new set of scenarios**, rather than a rewrite of the previous 28-problem worksheet. It follows the supplied tutorial's rhythm: a short explanation → a focused executable experiment → a small conceptual conclusion → the next experiment. The subject remains Python's built-in `object` class and the object model that grows out of it.

**Prerequisites:** classes, instances, functions, `is`, `==`, and dictionaries. **Runtime:** Python 3; standard library only. Run the cells **top to bottom**. Each problem uses locally named classes/variables so its experiments do not accidentally depend on a prior problem. Cells use portable assertions rather than memory addresses, interpreter-specific byte counts, or brittle exact exception text.

**Study routine:** before each experiment, predict the printout; run it; explain why; then change one assumption and rerun. Expected behaviors and complete worked solutions are included, not hidden.

### Road map

- Problems 1–5: types, instances, sentinels, namespace lookup, and object storage.
- Problems 6–10: inherited implementations, display, formatting, equality, and ordering.
- Problems 11–14: truth, identity hashing, attribute hooks, and controlled mutation.
- Problems 15–17: introspection, sizes, and cooperative class creation.
- Problem 18: a cohesive event-record capstone with an executable test suite.

**A precision note about the original tutorial:** default `object` equality is *identity-based in effect*, but the direct method `object.__eq__` can return `NotImplemented` for distinct operands. The `==` operator then runs the rich-comparison fallback. We will observe both behaviors rather than equating a method's return value with the entire operator protocol.

## Warm-up — Confirm the environment

`object` and `type` are built-ins. We inspect their relationships without relying on the particular address or string rendering of an instance.

In [1]:
import sys
import types

print("Python:", sys.version.split()[0])
assert isinstance(object, type)
assert isinstance(type, type)
assert issubclass(type, object)
print("Warm-up checks passed")

Python: 3.13.7
Warm-up checks passed


## Problem 01 — Separate three different questions

    **Challenge.** Given a class, an instance, and `type`, distinguish “what created it?”, “is it an instance?”, and “does it inherit from `object`?”

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Predict the graph

    `type(C)` asks about the class object `C`; `type(C())` asks about an instance. `issubclass` only accepts classes, whereas `isinstance` checks objects. Predict the next values before running.

In [2]:
class P01_Gadget:
    pass

p01_item = P01_Gadget()
print(type(P01_Gadget) is type)
print(type(p01_item) is P01_Gadget)
print(isinstance(P01_Gadget, object))
print(issubclass(P01_Gadget, object))

True
True
True
True


**Why this works.** All four lines are `True`. A class is itself an object (normally an instance of `type`); an instance of that class has the custom class as its type; and a normal Python class inherits `object` implicitly.

### Step 2 — Investigate the apparently circular part

    The root class is an instance of `type`, while `type` itself is a subclass of `object`. This is built into Python’s metaclass model, not a recursion that users need to implement.

In [3]:
p01_relationships = {
    "type(object) is type": type(object) is type,
    "type(type) is type": type(type) is type,
    "issubclass(type, object)": issubclass(type, object),
    "isinstance(object, object)": isinstance(object, object),
}
print(p01_relationships)
assert all(p01_relationships.values())

{'type(object) is type': True, 'type(type) is type': True, 'issubclass(type, object)': True, 'isinstance(object, object)': True}


### Step 3 — Solve the API-selection problem

    Write a classifier that answers the two questions separately. Validate its input instead of calling `issubclass` on arbitrary values.

In [4]:
def p01_class_report(candidate):
    if not isinstance(candidate, type):
        raise TypeError("expected a class, not an instance")
    return {
        "metaclass": type(candidate),
        "subclass_of_object": issubclass(candidate, object),
    }

assert p01_class_report(P01_Gadget)["metaclass"] is type
assert p01_class_report(P01_Gadget)["subclass_of_object"]
try:
    p01_class_report(p01_item)
except TypeError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("the instance must be rejected")

Expected error: expected a class, not an instance


**Takeaway.** `type(x)`, `isinstance(x, T)`, and `issubclass(C, T)` answer different questions. A class object and an ordinary instance are both objects.

---

## Problem 02 — Build a taxonomy of callable and module objects

    **Challenge.** Determine which objects are functions, built-in functions, and modules without mistaking callability for type identity.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Start with three objects

    The original tutorial inspects the `math` module and `types.FunctionType`. Add a built-in function to complicate matters. Predict whether the three types match.

In [5]:
import math

def p02_square(n):
    return n * n

p02_samples = (math, p02_square, len)
for sample in p02_samples:
    print(type(sample).__name__, "callable?", callable(sample))

module callable? False
function callable? True
builtin_function_or_method callable? True


**Why this works.** `math` is a module and not callable; `p02_square` is a Python function; `len` is a built-in function. The last two are both callable but have different concrete types.

### Step 2 — Use documented type aliases

    The `types` module exposes names for many built-in runtime types. Test identities directly rather than comparing printed class names.

In [6]:
assert type(math) is types.ModuleType
assert type(p02_square) is types.FunctionType
assert type(len) is types.BuiltinFunctionType
assert all(isinstance(value, object) for value in p02_samples)
print("Three concrete types; three objects")

Three concrete types; three objects


### Step 3 — Solve a realistic classifier

    We explicitly distinguish modules and Python functions, then use a broad callable category for other callable values. A callable class instance is not necessarily a function.

In [7]:
class P02_Worker:
    def __call__(self, n):
        return n + 1

def p02_describe(value):
    if isinstance(value, types.ModuleType):
        return "module"
    if isinstance(value, types.FunctionType):
        return "python function"
    if callable(value):
        return "other callable"
    return "other object"

assert [p02_describe(x) for x in (math, p02_square, len, P02_Worker(), 7)] == [
    "module", "python function", "other callable", "other callable", "other object"
]
print("Classification checks passed")

Classification checks passed


**Why this works.** `callable()` describes a capability, not a type. This is why checking a method or function’s exact class is often less flexible than checking the operation you actually need.

**Takeaway.** Modules, functions, classes, and callable instances are all objects, but “callable” does not imply “function”.

---

## Problem 03 — Design a missing-value sentinel that cannot confuse `None` and zero

    **Challenge.** Implement a mapping reader that distinguishes absent keys from explicitly stored `None`, `False`, `0`, and empty strings.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Find the bug

    A default of `None` loses information when `None` is itself valid data. A truthiness check also rejects valid false-like data.

In [8]:
p03_options = {"timeout": 0, "note": None, "enabled": False, "name": ""}
print(p03_options.get("note") is p03_options.get("absent"))
print(bool(p03_options["timeout"]))

True
False


**Why this works.** The first result is `True`, although one key exists and the other does not. The second result is `False`, even though the stored timeout is valid.

### Step 2 — Create one unique object

    `object()` provides a minimal identity-based marker. Its purpose is *not* to compare equal to another separately created marker.

In [9]:
P03_MISSING = object()
p03_another_missing = object()
assert P03_MISSING is not p03_another_missing
assert P03_MISSING != p03_another_missing
print("Two marker instances are distinct")

Two marker instances are distinct


### Step 3 — Write the complete reader

    Use the *same* sentinel for the `get` default and the identity check. Return the stored value unchanged even when it is false-like.

In [10]:
def p03_lookup(mapping, key):
    value = mapping.get(key, P03_MISSING)
    if value is P03_MISSING:
        raise KeyError(key)
    return value

for key, expected in (("timeout", 0), ("note", None),
                      ("enabled", False), ("name", "")):
    assert p03_lookup(p03_options, key) == expected
try:
    p03_lookup(p03_options, "absent")
except KeyError as exc:
    assert exc.args == ("absent",)
else:
    raise AssertionError("an absent key must raise")
print("All present/absent cases passed")

All present/absent cases passed


### Step 4 — Investigate sentinel scope

    A brand-new `object()` inside each call is *not* interchangeable with a module-level marker. Reusing a single sentinel is the invariant that matters.

In [11]:
def p03_is_missing(value):
    return value is P03_MISSING

assert p03_is_missing(P03_MISSING)
assert not p03_is_missing(object())
assert not p03_is_missing(None)
print("Identity-based sentinel checks passed")

Identity-based sentinel checks passed


**Takeaway.** Prefer `is` for sentinel identity. Do not use truthiness, `None`, or a newly allocated marker to stand for an absent but otherwise valid value.

---

## Problem 04 — Debug class-state leakage and attribute shadowing

    **Challenge.** A configuration value should default at class level but allow a per-instance override without affecting other instances.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Trace the lookup order with ordinary attributes

    A class variable is accessible through instances. Assigning the same name to one instance usually creates an entry in that instance’s dictionary; it does not rewrite the class dictionary.

In [12]:
class P04_Job:
    retries = 3

p04_a, p04_b = P04_Job(), P04_Job()
print(p04_a.retries, p04_b.retries, P04_Job.retries)
p04_a.retries = 9
print(p04_a.retries, p04_b.retries, P04_Job.retries)

3 3 3
9 3 3


**Why this works.** The outputs change from `3 3 3` to `9 3 3`. This is instance shadowing. The class attribute remains the default for other instances.

### Step 2 — Prove where the two values live

    Inspect namespace dictionaries rather than using `id()` values, which say nothing about *where a name is stored*.

In [13]:
assert P04_Job.__dict__["retries"] == 3
assert p04_a.__dict__ == {"retries": 9}
assert p04_b.__dict__ == {}
assert object.__getattribute__(p04_b, "retries") == 3
print("Class and instance namespaces verified")

Class and instance namespaces verified


### Step 3 — Find the real mutable-default trap

    Mutating a list obtained through an instance changes the shared class-level list. Merely assigning an integer attribute in Step 1 did not do that.

In [14]:
class P04_BadLog:
    events = []

p04_first, p04_second = P04_BadLog(), P04_BadLog()
p04_first.events.append("started")
assert p04_second.events == ["started"]
print("Observed shared list:", p04_second.events)

Observed shared list: ['started']


### Step 4 — Solve it using per-instance storage

    Initialize mutable state in `__init__` and use a class attribute only for an immutable shared default.

In [15]:
class P04_GoodLog:
    level = "INFO"

    def __init__(self):
        self.events = []

p04_x, p04_y = P04_GoodLog(), P04_GoodLog()
p04_x.events.append("started")
assert p04_x.events == ["started"] and p04_y.events == []
assert p04_x.level == p04_y.level == "INFO"
print("Mutable state is independent")

Mutable state is independent


**Takeaway.** Reading a class attribute, shadowing it with assignment, and mutating a shared mutable value are three different operations.

---

## Problem 05 — Compare three namespaces without assuming every object has a dictionary

    **Challenge.** Build a portable introspection function for modules, ordinary instances, and class objects.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Inspect the apparent symmetry

    Modules and conventional class instances have writable namespaces. Class objects expose a read-only view of their namespace through `__dict__`. Plain `object()` instances do not supply an instance dictionary.

In [16]:
class P05_Box:
    label = "default"

p05_box = P05_Box()
p05_box.color = "blue"
print(type(vars(math)).__name__)
print(type(vars(p05_box)).__name__)
print(type(P05_Box.__dict__).__name__)
print(hasattr(object(), "__dict__"))

dict
dict
mappingproxy
False


**Why this works.** Expect `dict`, `dict`, `mappingproxy`, and `False`. `mappingproxy` blocks mutation through the view, though normal class attribute assignment can still update the underlying class namespace.

### Step 2 — Confirm `vars` has a precondition

    Bare `object()` instances have no custom attribute storage by default. Show a controlled `TypeError` instead of assuming `vars` always works.

In [17]:
try:
    vars(object())
except TypeError:
    print("No __dict__ on plain object()")
else:
    raise AssertionError("vars(object()) should fail")

assert vars(p05_box)["color"] == "blue"
assert P05_Box.__dict__["label"] == "default"

No __dict__ on plain object()


### Step 3 — Solve a safe namespace browser

    This utility does not pretend `dir()` and `vars()` are equivalent: `dir()` lists discoverable names; `vars()` exposes actual dictionary state where it exists.

In [18]:
def p05_own_state(value):
    namespace = getattr(value, "__dict__", None)
    if namespace is None:
        return None
    return dict(namespace)

assert p05_own_state(p05_box) == {"color": "blue"}
assert p05_own_state(object()) is None
assert p05_own_state(P05_Box)["label"] == "default"
print("Namespace browser works for all three cases")

Namespace browser works for all three cases


**Takeaway.** A namespace view, actual instance state, and names exposed for discovery are related but not interchangeable.

---

## Problem 06 — Find exactly which class supplied a method

    **Challenge.** An instance has a method, but its class does not define it locally. Identify the defining class without comparing ephemeral object IDs.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Create a two-level inheritance chain

    Method lookup walks the method resolution order (MRO). Seeing a method through `Child.method` does not mean the child defined it.

In [19]:
class P06_Parent:
    def diagnose(self):
        return "from parent"

class P06_Child(P06_Parent):
    pass

print(P06_Child().diagnose())
print([cls.__name__ for cls in P06_Child.__mro__])

from parent
['P06_Child', 'P06_Parent', 'object']


### Step 2 — Distinguish local versus inherited definitions

    A class’s `__dict__` contains names defined on that class itself. Attribute access on the class performs inheritance lookup.

In [20]:
assert "diagnose" not in P06_Child.__dict__
assert "diagnose" in P06_Parent.__dict__
assert P06_Child.diagnose is P06_Parent.__dict__["diagnose"]
assert P06_Child.__repr__ is object.__repr__
print("Local and inherited definitions separated")

Local and inherited definitions separated


### Step 3 — Solve a reusable inspector

    Walk `__mro__` to locate the first namespace containing the name. This exercise handles ordinary inherited methods; custom metaclass lookups and descriptors require extra considerations.

In [21]:
def p06_defining_class(cls, name):
    for base in cls.__mro__:
        if name in vars(base):
            return base
    raise AttributeError(name)

assert p06_defining_class(P06_Child, "diagnose") is P06_Parent
assert p06_defining_class(P06_Child, "__repr__") is object
try:
    p06_defining_class(P06_Child, "never_defined")
except AttributeError:
    print("Undefined name correctly rejected")
else:
    raise AssertionError("missing definition should raise")

Undefined name correctly rejected


**Takeaway.** Inspect `__mro__` and each class’s own `__dict__` when asking “which class defined this method?”

---

## Problem 07 — Reconstruct the `str` / `repr` fallback chain

    **Challenge.** Predict representations for classes that override only one of the two methods, then design a useful representation pair.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Start at `object`

    The inherited `object.__str__` delegates to `repr(self)`. Therefore overriding only `__repr__` also affects `str()` when `__str__` is still inherited.

In [22]:
class P07_ReprOnly:
    def __repr__(self):
        return "P07_ReprOnly(debug=True)"

p07_a = P07_ReprOnly()
assert repr(p07_a) == "P07_ReprOnly(debug=True)"
assert str(p07_a) == "P07_ReprOnly(debug=True)"
print("repr:", repr(p07_a), "| str:", str(p07_a))

repr: P07_ReprOnly(debug=True) | str: P07_ReprOnly(debug=True)


### Step 2 — Reverse the override

    Overriding only `__str__` changes the user-facing string, but `repr()` still uses the generic inherited implementation. Do not assert the memory address or the whole default string.

In [23]:
class P07_StrOnly:
    def __str__(self):
        return "friendly name"

p07_b = P07_StrOnly()
assert str(p07_b) == "friendly name"
assert "P07_StrOnly" in repr(p07_b)
assert repr(p07_b) != str(p07_b)
print("str override does not replace repr")

str override does not replace repr


### Step 3 — Implement both with different audiences

    Make `repr` unambiguous for developers and `str` concise for humans. Returning a non-string from either hook is an error.

In [24]:
class P07_Ticket:
    def __init__(self, number, title):
        self.number = number
        self.title = title

    def __repr__(self):
        return f"P07_Ticket(number={self.number!r}, title={self.title!r})"

    def __str__(self):
        return f"#{self.number}: {self.title}"

p07_t = P07_Ticket(12, "Fix cache")
assert repr(p07_t) == "P07_Ticket(number=12, title='Fix cache')"
assert str(p07_t) == "#12: Fix cache"
print(repr(p07_t))
print(str(p07_t))

P07_Ticket(number=12, title='Fix cache')
#12: Fix cache


### Step 4 — Test an invalid override

    Magic methods have contracts: returning the wrong type is not silently converted into a string.

In [25]:
class P07_Broken:
    def __repr__(self):
        return 42

try:
    repr(P07_Broken())
except TypeError:
    print("Invalid __repr__ result rejected")
else:
    raise AssertionError("repr must reject a non-string")

Invalid __repr__ result rejected


**Takeaway.** `repr` is oriented toward debugging; `str` is oriented toward display. The inherited `__str__` dynamically uses `repr`.

---

## Problem 08 — Treat formatting as its own protocol

    **Challenge.** Make a report field support an empty format specification and one explicit nonempty specification without relying on unspecified default behavior.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Probe the inherited implementation

    In modern Python, `object.__format__` supports an empty format specification by using `str(self)`; a nonempty spec is rejected. The *exact exception text* is not part of our test.

In [26]:
class P08_Metric:
    def __str__(self):
        return "8 units"

p08_m = P08_Metric()
assert format(p08_m, "") == "8 units"
try:
    format(p08_m, ">12")
except TypeError:
    print("Inherited format rejects a nonempty spec")
else:
    raise AssertionError("expected unsupported format error")

Inherited format rejects a nonempty spec


### Step 2 — Define an explicit mini-language

    Support `""` for normal display and `"value"` for a machine-readable count. Reject all other specifications deliberately.

In [27]:
class P08_Count:
    def __init__(self, value):
        self.value = int(value)

    def __str__(self):
        return f"{self.value} items"

    def __format__(self, spec):
        if spec == "":
            return str(self)
        if spec == "value":
            return str(self.value)
        raise ValueError(f"unsupported format spec: {spec!r}")

p08_count = P08_Count(8)
assert f"{p08_count}" == "8 items"
assert f"{p08_count:value}" == "8"
print(f"human: {p08_count} | machine: {p08_count:value}")

human: 8 items | machine: 8


### Step 3 — Test failures and conversion flags

    An f-string conversion like `!r` calls `repr` first, so formatting then applies to the resulting string; it does not call the object’s custom `__format__` method.

In [28]:
try:
    format(p08_count, "unknown")
except ValueError:
    print("Unknown custom format rejected")
else:
    raise AssertionError("unsupported specification must raise")

assert f"{p08_count!r}" == repr(p08_count)
assert format(p08_count, "value") == "8"

Unknown custom format rejected


**Takeaway.** `format(x, spec)` is a separate protocol. Define and document accepted specifications instead of assuming `str` handles arbitrary format strings.

---

## Problem 09 — Distinguish the result of `object.__eq__` from `==`

    **Challenge.** Explain exactly why two new `object()` instances compare unequal although a direct call to `object.__eq__` may not return `False`.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Observe the two layers

    The direct method signals that it cannot decide equality for distinct objects; the rich-comparison operator then tries its remaining path and ultimately produces a Boolean.

In [29]:
p09_left, p09_right = object(), object()
assert object.__eq__(p09_left, p09_left) is True
assert object.__eq__(p09_left, p09_right) is NotImplemented
assert (p09_left == p09_right) is False
assert (p09_left != p09_right) is True
print("Direct method:", object.__eq__(p09_left, p09_right))
print("Operator:", p09_left == p09_right)

Direct method: NotImplemented
Operator: False


**Why this works.** `NotImplemented` is a protocol value, not a raised exception and not a false Boolean. When neither operand implements equality for the pair, equality falls back to identity-based results.

### Step 2 — Implement a strict domain equality method

    Compare equivalent `Code` objects by value, but return `NotImplemented` for unrelated classes so the other operand gets an opportunity to respond.

In [30]:
class P09_Code:
    def __init__(self, text):
        self.text = text

    def __eq__(self, other):
        if not isinstance(other, P09_Code):
            return NotImplemented
        return self.text == other.text

p09_a, p09_b, p09_c = P09_Code("A"), P09_Code("A"), P09_Code("B")
assert p09_a == p09_b
assert p09_a != p09_c
assert P09_Code.__eq__(p09_a, "A") is NotImplemented
assert (p09_a == "A") is False
print("Value equality and unknown-type fallback verified")

Value equality and unknown-type fallback verified


### Step 3 — Check the hash consequence

    In Python, defining `__eq__` on a class without a corresponding compatible `__hash__` normally makes instances unhashable. This prevents accidentally putting changeable value-equality objects into sets.

In [31]:
assert P09_Code.__hash__ is None
try:
    hash(p09_a)
except TypeError:
    print("Value-equality object is unhashable by default")
else:
    raise AssertionError("this class must be unhashable")

Value-equality object is unhashable by default


**Takeaway.** The operator protocol, not a direct `__eq__` invocation, determines `a == b`. Return `NotImplemented` for an unsupported operand type.

---

## Problem 10 — Add ordering without inventing an order for unrelated objects

    **Challenge.** Sort a list of priority records while refusing to compare a record with a number.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Identify the missing feature of `object`

    Plain objects do not provide a meaningful less-than ordering. Their inherited rich comparison returns `NotImplemented` for `<`, and the operator raises `TypeError` if neither side can handle it.

In [32]:
p10_a, p10_b = object(), object()
assert object.__lt__(p10_a, p10_b) is NotImplemented
try:
    p10_a < p10_b
except TypeError:
    print("Plain objects have no intrinsic ordering")
else:
    raise AssertionError("unrelated objects should not be ordered")

Plain objects have no intrinsic ordering


### Step 2 — Choose a defensible ordering contract

    A priority is a number; two records are ordered by `(priority, name)`. Both comparison methods check the operand type and defer unrelated objects.

In [33]:
from functools import total_ordering

@total_ordering
class P10_Task:
    def __init__(self, priority, name):
        self.priority = priority
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, P10_Task):
            return NotImplemented
        return (self.priority, self.name) == (other.priority, other.name)

    def __lt__(self, other):
        if not isinstance(other, P10_Task):
            return NotImplemented
        return (self.priority, self.name) < (other.priority, other.name)

    def __repr__(self):
        return f"P10_Task({self.priority!r}, {self.name!r})"

p10_tasks = [P10_Task(2, "email"), P10_Task(1, "review"), P10_Task(1, "build")]
print(sorted(p10_tasks))

[P10_Task(1, 'build'), P10_Task(1, 'review'), P10_Task(2, 'email')]


### Step 3 — Verify generated operators and error cases

    `total_ordering` fills in the remaining ordering operators; the two comparison methods remain the explicit source of truth. This decorator does not supply a hash function.

In [34]:
p10_sorted = sorted(p10_tasks)
assert [x.name for x in p10_sorted] == ["build", "review", "email"]
assert P10_Task(1, "build") <= P10_Task(1, "build")
assert P10_Task(2, "a") > P10_Task(1, "z")
assert P10_Task.__lt__(p10_tasks[0], 9) is NotImplemented
try:
    p10_tasks[0] < 9
except TypeError:
    print("Cross-type order correctly rejected")
else:
    raise AssertionError("invalid cross-type order must raise")
assert P10_Task.__hash__ is None

Cross-type order correctly rejected


**Takeaway.** `object` supplies no natural ordering. Define a domain-specific key, return `NotImplemented` for unrelated types, and test all relevant operators.

---

## Problem 11 — Examine how truthiness starts at the `object` default

    **Challenge.** Specify when a custom container-like object should be considered false, and catch two invalid truth-protocol implementations.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Establish the default

    An ordinary instance is truthy even if it contains fields such as `count = 0`. Python first considers `__bool__`, then `__len__`, and otherwise treats the instance as true.

In [35]:
class P11_Empty:
    count = 0

assert bool(P11_Empty()) is True
assert bool(object()) is True
print("Plain instances are truthy")

Plain instances are truthy


### Step 2 — Prefer a length-based protocol for collections

    If a container naturally has a length, implementing `__len__` makes both `len(x)` and `bool(x)` consistent without redundant code.

In [36]:
class P11_Bag:
    def __init__(self, items=()):
        self.items = tuple(items)

    def __len__(self):
        return len(self.items)

p11_empty, p11_full = P11_Bag(), P11_Bag([1, 2])
assert not p11_empty and p11_full
assert len(p11_empty) == 0 and len(p11_full) == 2
print("Empty?", bool(p11_empty), "Full?", bool(p11_full))

Empty? False Full? True


### Step 3 — Override only when the domain requires it

    `__bool__` takes precedence over `__len__`. A record can be “inactive” even though it has stored items.

In [37]:
class P11_ActiveBag(P11_Bag):
    def __init__(self, items=(), active=True):
        super().__init__(items)
        self.active = active

    def __bool__(self):
        return self.active and len(self) > 0

p11_disabled = P11_ActiveBag([1], active=False)
assert len(p11_disabled) == 1
assert bool(p11_disabled) is False
print("Length and activity intentionally differ")

Length and activity intentionally differ


### Step 4 — Test protocol constraints

    `__bool__` must return a real `bool`; `__len__` must produce a nonnegative integer. Bad hooks cause exceptions rather than being silently coerced.

In [38]:
class P11_BadBool:
    def __bool__(self):
        return 1

class P11_BadLength:
    def __len__(self):
        return -1

for instance, expected in ((P11_BadBool(), TypeError),
                           (P11_BadLength(), ValueError)):
    try:
        bool(instance)
    except expected:
        print(expected.__name__, "detected")
    else:
        raise AssertionError("expected protocol failure")

TypeError detected
ValueError detected


**Takeaway.** Default object truthiness is `True`. For collections, `__len__` often suffices; explicit `__bool__` overrides it and must return `bool`.

---

## Problem 12 — Use identity-based objects as safe registry tokens

    **Challenge.** Store metadata under object keys while allowing metadata to change, and prove why a different token with identical metadata is still a different key.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Understand the default hash model

    Ordinary classes that do not override equality inherit identity-based equality and compatible identity-based hashing from `object`. A hash is *not* a guaranteed unique identifier, and we do not need to inspect its numeric value.

In [39]:
class P12_Token:
    def __init__(self, title):
        self.title = title

p12_t1, p12_t2 = P12_Token("shared"), P12_Token("shared")
assert p12_t1 is not p12_t2
assert p12_t1 != p12_t2
assert P12_Token.__hash__ is object.__hash__
assert isinstance(hash(p12_t1), int)
print("Distinct tokens can share metadata")

Distinct tokens can share metadata


### Step 2 — Solve the registry problem

    A dictionary key is an identity token; metadata belongs in the associated value or in fields whose mutation does not affect identity-based hashing.

In [40]:
p12_registry = {p12_t1: {"status": "queued"}}
p12_t1.title = "renamed"
p12_registry[p12_t1]["status"] = "done"
assert p12_registry[p12_t1]["status"] == "done"
assert p12_t2 not in p12_registry
assert len(p12_registry) == 1
print("Renaming metadata did not change the dictionary key")

Renaming metadata did not change the dictionary key


### Step 3 — Demonstrate why metadata equality changes the design

    For an ordinary class, writing `__eq__` without `__hash__` automatically disables hashing. A value-key design would need a compatible hash based only on stable values; that is a *different* requirement from identity tokens.

In [41]:
class P12_ValueToken:
    def __init__(self, title):
        self.title = title

    def __eq__(self, other):
        if not isinstance(other, P12_ValueToken):
            return NotImplemented
        return self.title == other.title

assert P12_ValueToken.__hash__ is None
try:
    {P12_ValueToken("x"): 1}
except TypeError:
    print("Value-equality tokens reject dictionary use")
else:
    raise AssertionError("unhashable key was accepted")

Value-equality tokens reject dictionary use


**Takeaway.** Choose identity semantics deliberately for token objects. A mutable descriptive field does not change object identity; changing equality semantics changes the hashing contract.

---

## Problem 13 — Understand when `__getattr__` actually runs

    **Challenge.** Build a computed fallback field and distinguish ordinary access from a direct call to `object.__getattribute__`.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Create a normal lookup with a fallback

    Python invokes `__getattr__` when ordinary attribute lookup raises `AttributeError`. Existing attributes do not use it.

In [42]:
class P13_Profile:
    def __init__(self, given, family):
        self.given = given
        self.family = family

    def __getattr__(self, name):
        if name == "full_name":
            return f"{self.given} {self.family}"
        raise AttributeError(name)

p13_profile = P13_Profile("Ada", "Lovelace")
assert p13_profile.given == "Ada"
assert p13_profile.full_name == "Ada Lovelace"
print(p13_profile.full_name)

Ada Lovelace


### Step 2 — Call the root lookup directly

    `object.__getattribute__` is the primitive lookup implementation; directly calling it is *not* the same as normal attribute access with the language-level `__getattr__` fallback.

In [43]:
assert object.__getattribute__(p13_profile, "given") == "Ada"
try:
    object.__getattribute__(p13_profile, "full_name")
except AttributeError:
    print("Direct primitive lookup does not invoke fallback")
else:
    raise AssertionError("computed name should not be stored")

Direct primitive lookup does not invoke fallback


### Step 3 — Make failure semantics explicit

    `getattr(obj, name, default)` returns the default on `AttributeError`; `hasattr` also tests for that specific exception. Never swallow unrelated exceptions in fallback code.

In [44]:
P13_ABSENT = object()
assert getattr(p13_profile, "full_name", P13_ABSENT) == "Ada Lovelace"
assert getattr(p13_profile, "nickname", P13_ABSENT) is P13_ABSENT
assert hasattr(p13_profile, "full_name")
assert not hasattr(p13_profile, "nickname")
assert "full_name" not in vars(p13_profile)
print("Virtual versus stored attributes distinguished")

Virtual versus stored attributes distinguished


### Step 4 — Fix a problematic fallback

    A `__getattr__` implementation should raise `AttributeError` for unknown names rather than returning `None`, which would make `hasattr` incorrectly report that every name exists.

In [45]:
class P13_BrokenProfile:
    def __getattr__(self, name):
        return None

assert hasattr(P13_BrokenProfile(), "anything") is True
assert hasattr(p13_profile, "anything") is False
print("The corrected class reports missing names accurately")

The corrected class reports missing names accurately


**Takeaway.** `__getattr__` is a fallback for missing names, not an alias for `object.__getattribute__`. Preserve `AttributeError` for truly absent attributes.

---

## Problem 14 — Build a write-once configuration safely

    **Challenge.** Allow initialization, then refuse attribute reassignment or deletion, while showing the power and limits of `object.__setattr__`.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Identify the recursive assignment trap

    Inside an override of `__setattr__`, writing `self.name = value` invokes the override again. Delegate the actual store to the base implementation instead.

In [46]:
class P14_WriteOnce:
    def __init__(self, value):
        object.__setattr__(self, "value", value)
        object.__setattr__(self, "_locked", True)

    def __setattr__(self, name, value):
        if getattr(self, "_locked", False):
            raise AttributeError("configuration is write-once")
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        if getattr(self, "_locked", False):
            raise AttributeError("configuration is write-once")
        object.__delattr__(self, name)

p14_setting = P14_WriteOnce(17)
assert p14_setting.value == 17
print("Initialized:", p14_setting.value)

Initialized: 17


### Step 2 — Test public mutation paths

    An assignment and a deletion should each raise. We test the error *type* and relevant final state, not brittle wording.

In [47]:
for action in (lambda: setattr(p14_setting, "value", 19),
               lambda: delattr(p14_setting, "value")):
    try:
        action()
    except AttributeError:
        print("Mutation blocked")
    else:
        raise AssertionError("mutation should have been blocked")
assert p14_setting.value == 17

Mutation blocked
Mutation blocked


### Step 3 — State the security boundary accurately

    This is an API-level guard, **not a security boundary**: a caller can explicitly invoke `object.__setattr__` on a normal instance and bypass the override. Use stronger encapsulation or process boundaries for security requirements.

In [48]:
p14_demo = P14_WriteOnce(1)
object.__setattr__(p14_demo, "value", 2)  # Intentional bypass demonstration
assert p14_demo.value == 2
print("Direct base-method call bypassed the API guard")

Direct base-method call bypassed the API guard


### Step 4 — Write a design decision

    Use this pattern for accidental mutation prevention, not secrecy or adversarial protection. If nested fields contain lists or dictionaries, freezing just the attributes does not freeze those inner objects.

In [49]:
p14_nested = P14_WriteOnce(["a"])
p14_nested.value.append("b")
assert p14_nested.value == ["a", "b"]
print("Nested mutable value remains mutable")

Nested mutable value remains mutable


**Takeaway.** Calling `object.__setattr__` or `object.__delattr__` avoids recursive hooks but bypasses the ordinary guard. Attribute freezing is not deep immutability or security.

---

## Problem 15 — Customize discoverability without confusing it with access control

    **Challenge.** Expose a virtual attribute in `dir()` while ensuring the attribute is actually implemented and no secret is treated as protected merely because it is hidden.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Understand the default

    `dir(instance)` tries to report useful discoverable names; it is not a promise that every result is stored in `vars(instance)`, nor a list of all possible dynamic names.

In [50]:
class P15_Record:
    def __init__(self, code):
        self.code = code

    def __getattr__(self, name):
        if name == "display":
            return f"Record[{self.code}]"
        raise AttributeError(name)

p15_record = P15_Record("A7")
assert p15_record.display == "Record[A7]"
print("display in dir before override?", "display" in dir(p15_record))

display in dir before override? False


### Step 2 — Implement a helpful `__dir__`

    Delegate to `object.__dir__` so inherited/standard names are retained, then add the public virtual name. Sorting makes output stable for readers and test assertions.

In [51]:
class P15_Discoverable(P15_Record):
    def __dir__(self):
        return sorted(set(object.__dir__(self)) | {"display"})

p15_item = P15_Discoverable("B2")
assert "display" in dir(p15_item)
assert "code" in dir(p15_item)
assert p15_item.display == "Record[B2]"
print("Virtual name advertised:", "display" in dir(p15_item))

Virtual name advertised: True


### Step 3 — Challenge a mistaken security assumption

    Removing a name from `dir()` cannot prohibit normal attribute access. Use a tiny demonstration; deliberately avoid claiming this is an access-control mechanism.

In [52]:
class P15_Hidden:
    def __init__(self):
        self.secret = "readable"

    def __dir__(self):
        return ["public"]

p15_hidden = P15_Hidden()
assert "secret" not in dir(p15_hidden)
assert p15_hidden.secret == "readable"
print("Hidden from dir, still directly accessible")

Hidden from dir, still directly accessible


**Takeaway.** `__dir__` affects interactive discovery only. Actual availability is governed by attribute lookup, not whether a name appears in `dir()`.

---

## Problem 16 — Measure an object without claiming `__sizeof__` is a deep-memory profiler

    **Challenge.** Explain why the direct size of an object can differ from its size reported by `sys.getsizeof`, and why neither automatically counts referenced data.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Collect sizes without hard-coded byte counts

    `object.__sizeof__` reports an implementation-specific size estimate for the object itself. `sys.getsizeof` may add garbage-collector overhead. Neither reports a full transitive graph size.

In [53]:
import sys

class P16_Holder:
    def __init__(self, items):
        self.items = items

p16_small = P16_Holder([])
p16_large = P16_Holder(list(range(10_000)))
p16_data = {
    "direct_small": p16_small.__sizeof__(),
    "reported_small": sys.getsizeof(p16_small),
    "direct_large": p16_large.__sizeof__(),
    "reported_large": sys.getsizeof(p16_large),
    "contained_list": sys.getsizeof(p16_large.items),
}
print(p16_data)

{'direct_small': 16, 'reported_small': 48, 'direct_large': 16, 'reported_large': 48, 'contained_list': 80056}


### Step 2 — Infer the key limit

    The holder references an external list; changing the list’s length does not require the holder itself to grow. Do not assert exact byte counts because Python implementations and builds differ.

In [54]:
assert isinstance(p16_data["direct_small"], int)
assert isinstance(p16_data["reported_small"], int)
assert p16_data["reported_small"] >= p16_data["direct_small"]
assert sys.getsizeof(p16_large.items) >= sys.getsizeof(p16_small.items)
print("Portable size relationships verified")

Portable size relationships verified


### Step 3 — Solve a reporting task honestly

    Report the instance, its dictionary (when present), and its referenced list separately. **Do not sum them and call the result total usage**: shared references, nested objects, and allocator overhead can make that misleading.

In [55]:
def p16_shallow_report(holder):
    return {
        "instance_bytes": sys.getsizeof(holder),
        "namespace_bytes": sys.getsizeof(vars(holder)),
        "list_bytes": sys.getsizeof(holder.items),
        "scope": "separate shallow measurements; not total retained memory",
    }

p16_report = p16_shallow_report(p16_large)
assert all(p16_report[k] > 0 for k in ("instance_bytes", "namespace_bytes", "list_bytes"))
print(p16_report)

{'instance_bytes': 48, 'namespace_bytes': 288, 'list_bytes': 80056, 'scope': 'separate shallow measurements; not total retained memory'}


**Takeaway.** Object size APIs give shallow, implementation-dependent measurements. Avoid exact-byte assertions and avoid presenting a naïve sum as an object graph’s memory footprint.

---

## Problem 17 — Explain keyword arguments at the `object.__init_subclass__` boundary

    **Challenge.** Create a reusable base class that accepts its own class-definition option but detects unexpected class keywords rather than silently swallowing them.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### Step 1 — Reproduce the boundary failure

    An unconsumed class-definition keyword eventually reaching `object.__init_subclass__` causes a `TypeError`. A plain base class does not accept arbitrary class creation arguments.

In [56]:
class P17_PlainBase:
    pass

try:
    class P17_Invalid(P17_PlainBase, mode="fast"):
        pass
except TypeError:
    print("Unconsumed class keyword was rejected")
else:
    raise AssertionError("unknown class keyword should fail")

Unconsumed class keyword was rejected


### Step 2 — Consume your own keyword cooperatively

    `__init_subclass__` is called when a subclass is created. The base consumes only `mode`, stores it on the newly created class, then forwards any other keywords via `super()`.

In [57]:
class P17_ModeBase:
    def __init_subclass__(cls, *, mode="safe", **kwargs):
        super().__init_subclass__(**kwargs)
        if mode not in ("safe", "fast"):
            raise ValueError("mode must be safe or fast")
        cls.mode = mode

class P17_Default(P17_ModeBase):
    pass

class P17_Fast(P17_ModeBase, mode="fast"):
    pass

assert P17_Default.mode == "safe"
assert P17_Fast.mode == "fast"
print(P17_Default.mode, P17_Fast.mode)

safe fast


### Step 3 — Exercise validation and the forwarding contract

    An invalid known option is a domain error. An unexpected option is not consumed, so the cooperative chain reaches `object` and raises `TypeError`.

In [58]:
try:
    class P17_BadMode(P17_ModeBase, mode="turbo"):
        pass
except ValueError:
    print("Known keyword validated")
else:
    raise AssertionError("invalid mode should fail")

try:
    class P17_Unknown(P17_ModeBase, unexpected=True):
        pass
except TypeError:
    print("Unknown keyword rejected by cooperative chain")
else:
    raise AssertionError("unknown keyword should fail")

Known keyword validated
Unknown keyword rejected by cooperative chain


### Step 4 — Confirm ordinary subclass relationships

    A hook changes subclass creation behavior, not the fact that classes inherit from `object`. No metaclass was necessary here.

In [59]:
assert issubclass(P17_Fast, P17_ModeBase)
assert issubclass(P17_Fast, object)
assert isinstance(P17_Fast(), object)
print("Class-creation hook and ordinary inheritance coexist")

Class-creation hook and ordinary inheritance coexist


**Takeaway.** Consume only the keywords your class understands and forward the rest. The default root hook helps detect unrecognized class-definition options.

---

## Problem 18 — Capstone — An event envelope with deliberately identity-based semantics

    **Challenge.** Build an object that preserves missing-versus-`None`, supports clear display, exposes a derived attribute, prevents accidental reassignment, and serves as an identity key in a registry.

    **Workflow:** make a prediction, isolate a small experiment, explain the mechanism, then test the complete solution. The final assertion cell is a miniature regression test.

### The design brief

An event transport must distinguish `payload=None` (explicitly supplied) from *no payload supplied*. Separate event objects with identical fields still represent **different occurrences**, so their equality and hashing remain inherited from `object`. This differs intentionally from a value object.

**Requirements:**

1. A private `object()` sentinel marks an omitted payload.
2. A missing payload displays as `<missing>` but is distinguishable from explicit `None`.
3. A read-only `summary` is computed using ordinary lookup, not stored in the instance dictionary.
4. A helpful `repr`, `str`, and `dir` are supplied.
5. Ordinary attribute mutation/deletion is rejected; references to nested mutable payloads are *not* deep-frozen.
6. Events remain distinct dictionary keys even when their fields match.

### Step 1 — Prove the input ambiguity

    A default of `None` would collapse “not passed” and “passed `None`”. Keep a single sentinel at class level and compare it using `is`.

In [60]:
class P18_Event:
    _MISSING = object()

    def __init__(self, name, payload=_MISSING):
        if not isinstance(name, str) or not name:
            raise ValueError("event name must be a nonempty string")
        object.__setattr__(self, "name", name)
        object.__setattr__(self, "payload", payload)

    @property
    def has_payload(self):
        return self.payload is not type(self)._MISSING

p18_missing = P18_Event("ping")
p18_none = P18_Event("ping", None)
assert not p18_missing.has_payload
assert p18_none.has_payload
print("Absent and explicit None remain distinct")

Absent and explicit None remain distinct


**Why this works.** The stored sentinel is one specific object belonging to the class. The property checks identity rather than equality or truthiness, so `0`, `False`, and `None` all count as explicitly supplied values.

### Step 2 — Extend the class in small, inspectable increments

    For tutorial purposes we add methods after the initial definition. This also demonstrates that classes are objects whose attributes can be installed via normal class assignment. In a production class, define these methods in the class body.

In [61]:
def p18_repr(self):
    if not self.has_payload:
        return f"P18_Event(name={self.name!r})"
    return f"P18_Event(name={self.name!r}, payload={self.payload!r})"

def p18_str(self):
    return f"{self.name}: {self.payload!r}" if self.has_payload else f"{self.name}: <missing>"

P18_Event.__repr__ = p18_repr
P18_Event.__str__ = p18_str
assert repr(p18_missing) == "P18_Event(name='ping')"
assert repr(p18_none) == "P18_Event(name='ping', payload=None)"
assert str(p18_missing) == "ping: <missing>"
print(repr(p18_missing))
print(str(p18_none))

P18_Event(name='ping')
ping: None


### Step 3 — Add a derived attribute through `__getattr__`

    `summary` should be accessible but should not appear in `vars(event)`. Unknown names must still raise `AttributeError`.

In [62]:
def p18_getattr(self, name):
    if name == "summary":
        return f"{self.name} / {'supplied' if self.has_payload else 'missing'}"
    raise AttributeError(name)

P18_Event.__getattr__ = p18_getattr
assert p18_missing.summary == "ping / missing"
assert p18_none.summary == "ping / supplied"
assert "summary" not in vars(p18_missing)
assert not hasattr(p18_missing, "unknown")
print(p18_missing.summary)

ping / missing


### Step 4 — Make discovery useful

    The virtual name is implemented, so advertising it through `dir()` is truthful. The event also inherits the normal attribute methods supplied by `object`.

In [63]:
def p18_dir(self):
    return sorted(set(object.__dir__(self)) | {"summary"})

P18_Event.__dir__ = p18_dir
assert "summary" in dir(p18_missing)
assert "name" in dir(p18_missing)
assert "__class__" in dir(p18_missing)
print("Discoverability checks passed")

Discoverability checks passed


### Step 5 — Reject ordinary reassignment and deletion

    Initialization previously wrote via `object.__setattr__`. After that, these two hooks block normal mutation. Remember that such hooks are an API convention, not a security barrier.

In [64]:
def p18_setattr(self, name, value):
    raise AttributeError("event attributes cannot be reassigned")

def p18_delattr(self, name):
    raise AttributeError("event attributes cannot be deleted")

P18_Event.__setattr__ = p18_setattr
P18_Event.__delattr__ = p18_delattr
for action in (lambda: setattr(p18_missing, "name", "pong"),
               lambda: delattr(p18_missing, "payload")):
    try:
        action()
    except AttributeError:
        pass
    else:
        raise AssertionError("ordinary mutation should be blocked")
assert p18_missing.name == "ping"
print("Public mutation guards passed")

Public mutation guards passed


### Step 6 — Keep identity semantics on purpose

    Do **not** implement `__eq__` or `__hash__`: each event is an occurrence, not a value. Hashing and equality from `object` already meet that requirement.

In [65]:
p18_a = P18_Event("updated", 1)
p18_b = P18_Event("updated", 1)
assert p18_a is not p18_b and p18_a != p18_b
assert P18_Event.__eq__ is object.__eq__
assert P18_Event.__hash__ is object.__hash__
p18_index = {p18_a: "first", p18_b: "second"}
assert len(p18_index) == 2
assert p18_index[p18_b] == "second"
print("Two identical-looking events retain separate identities")

Two identical-looking events retain separate identities


### Step 7 — Test explicit false-like inputs and known limitations

    An event with `payload=0` or `payload=False` has a payload. A list payload can still be mutated from outside; attribute guards never promised deep immutability.

In [66]:
for value in (0, False, None, ""):
    assert P18_Event("data", value).has_payload
p18_list = [1]
p18_mutable = P18_Event("data", p18_list)
p18_list.append(2)
assert p18_mutable.payload == [1, 2]
print("False-like values and shallow immutability verified")

False-like values and shallow immutability verified


### Final solution audit

The implementation deliberately combines several `object` fundamentals: identity sentinels, base-method initialization, inherited identity equality/hash, user/debug text conversion, missing-attribute fallback, and honest `dir()` behavior. Its class-level method assignments were chosen to teach incremental inspection; define these methods directly in a single class declaration in real applications. The following end-to-end tests exercise the externally promised behavior.

In [67]:
def p18_test_contract():
    absent = P18_Event("a")
    explicit = P18_Event("a", None)
    assert absent.has_payload is False
    assert explicit.has_payload is True
    assert "<missing>" in str(absent)
    assert "payload=None" in repr(explicit)
    assert absent.summary == "a / missing"
    assert "summary" in dir(absent)
    assert "summary" not in vars(absent)
    assert absent != explicit
    assert len({absent, explicit}) == 2
    for invalid in ("", None, 4):
        try:
            P18_Event(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("invalid name accepted")
    try:
        absent.payload = 10
    except AttributeError:
        pass
    else:
        raise AssertionError("mutation accepted")
    return "Capstone: all contract tests passed"

print(p18_test_contract())

Capstone: all contract tests passed


**Takeaway.** Good object design starts by choosing the correct semantics: identity versus value, absent versus explicit data, public API versus security, and shallow versus deep immutability.

---

## Final review — Explain, do not just memorize

Use these prompts to check your understanding **without executing code first**:

1. Why is `type(MyClass)` a different question from `type(MyClass())`?
2. Why does a `len`-like built-in differ from a Python-defined function despite both being callable?
3. Why is `object()` useful as a missing-value sentinel but unsuitable as a universal, newly constructed replacement marker?
4. Why can assigning `instance.retries = 9` be safe while `instance.events.append(...)` can leak across instances?
5. Why might `vars(x)` fail even though `dir(x)` succeeds?
6. Why does `str(x)` use a custom `__repr__` when `__str__` is inherited?
7. Why might `object.__eq__(a, b)` return `NotImplemented` while `a == b` returns `False`?
8. Why does a class with new value equality become unhashable without a compatible `__hash__`?
9. Why does a direct `object.__getattribute__` call differ from ordinary fallback access?
10. Why are custom `__dir__` and `__setattr__` not security guarantees?
11. Why do `sys.getsizeof` and `__sizeof__` not measure a complete object graph?
12. Why can two event records with identical fields intentionally remain distinct dictionary keys?

**Answer key:** revisit the corresponding worked experiments: (1) P01, (2) P02, (3) P03, (4) P04, (5) P05, (6) P07, (7) P09, (8) P09/P12, (9) P13, (10) P14/P15, (11) P16, (12) P18.

**Next study technique:** change exactly one condition in each solved problem—for example, turn `None` into `0`, remove one method override, or introduce an unrelated operand type—and predict the behavior before running the code.